# TUBETU AI — бесплатный тест распознавания фотографий

Запускается в Google Colab. Использует открытую модель Hugging Face SmolVLM2-2.2B-Instruct. Это **тестовая среда**, а не постоянный сервер сайта: бесплатная сессия может отключаться, GPU доступен не всегда.

Не загружайте сюда личные документы или фотографии с конфиденциальными данными. Этот блокнот только проверяет модель и не публикует объявления.

In [ ]:
!pip -q install "transformers==4.49.0" accelerate pillow

In [ ]:
import torch
from PIL import Image
from google.colab import files
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Устройство:", device)
if device == "cpu":
    print("Предупреждение: без GPU модель может работать очень медленно или не поместиться в память.")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
).to(device)
model.eval()
print("Модель загружена.")

In [ ]:
# Выберите от 1 до 5 фотографий одного товара/объекта.
uploaded = files.upload()
if not 1 <= len(uploaded) <= 5:
    raise ValueError("Выберите от 1 до 5 фотографий.")
images = []
for name, data in uploaded.items():
    import io
    image = Image.open(io.BytesIO(data)).convert("RGB")
    image.thumbnail((1024, 1024))
    images.append(image)
print(f"Загружено фотографий: {len(images)}")

In [ ]:
prompt = """Проанализируй фотографии для черновика объявления на платформе TUBETU.
Верни только JSON с полями:
{"recognizedObject":"...","category":"...","title":"...","description":"...","visibleText":"...","confidence":"high|medium|low","unknowns":[]}
category выбери строго из списка: Проекты и идеи; Игры; Контент; Товары и ресурсы; Услуги; Логистика и перевозки; Финансы; Недвижимость; Транспорт; Сельское хозяйство; Техника и оборудование; Образование и опыт; Сообщества.
Опиши только то, что действительно видно. Не выдумывай цену, местоположение, состояние, бренд или модель. Если не уверен, укажи это в unknowns и confidence=low. Не публикуй объявление."""

content = [{"type": "image"} for _ in images]
content.append({"type": "text", "text": prompt})
messages = [{"role": "user", "content": content}]
formatted = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=formatted, images=images, return_tensors="pt")
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=400, do_sample=False)
new_tokens = output[0][inputs["input_ids"].shape[1]:]
result = processor.decode(new_tokens, skip_special_tokens=True)
print(result)

## Что проверяем
- Правильно ли определён объект и категория.
- Составлено ли короткое фактическое описание.
- Не выдуманы ли цена, бренд, состояние или адрес.

Если результат подходит, следующим шагом перенесём адаптер модели в `ai-core` и отдельно проверим интеграцию. Пока эта модель не подключена к действующему сайту.